In [3]:
# Импорт необходимых библиотек
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля для графиков
plt.style.use('seaborn-v0_8-darkgrid')

# Функции для различных моделей эпидемий

def sir_model(y, t, beta, gamma, N):
    """Модель SIR"""
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]

def seir_model(y, t, beta, sigma, gamma, N):
    """Модель SEIR"""
    S, E, I, R = y
    dSdt = -beta * S * I / N
    dEdt = beta * S * I / N - sigma * E
    dIdt = sigma * E - gamma * I
    dRdt = gamma * I
    return [dSdt, dEdt, dIdt, dRdt]

def sird_model(y, t, beta, gamma, mu, N):
    """Модель SIRD"""
    S, I, R, D = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I - mu * I
    dRdt = gamma * I
    dDdt = mu * I
    return [dSdt, dIdt, dRdt, dDdt]

def calculate_r0(beta, gamma, mu=None, model='SIR'):
    """Вычисление базового репродуктивного числа R0"""
    if model == 'SIR' or model == 'SEIR':
        return beta / gamma
    elif model == 'SIRD':
        return beta / (gamma + mu)

# Функция для создания интерактивной панели
def create_simulation():
    # Создание виджетов
    model_selector = widgets.Dropdown(
        options=['SIR', 'SEIR', 'SIRD'],
        value='SIR',
        description='Модель:',
        style={'description_width': 'initial'}
    )
    
    population = widgets.FloatSlider(
        value=10000,
        min=1000,
        max=100000,
        step=1000,
        description='Население:',
        style={'description_width': 'initial'}
    )
    
    initial_infected = widgets.FloatSlider(
        value=10,
        min=1,
        max=1000,
        step=1,
        description='Нач. зараженные:',
        style={'description_width': 'initial'}
    )
    
    initial_exposed = widgets.FloatSlider(
        value=5,
        min=0,
        max=500,
        step=1,
        description='Нач. экспонированные:',
        style={'description_width': 'initial'},
        disabled=True
    )
    
    beta = widgets.FloatSlider(
        value=0.3,
        min=0.01,
        max=1.0,
        step=0.01,
        description='β (контактность):',
        style={'description_width': 'initial'}
    )
    
    sigma = widgets.FloatSlider(
        value=0.2,
        min=0.01,
        max=1.0,
        step=0.01,
        description='σ (инкубация):',
        style={'description_width': 'initial'},
        disabled=True
    )
    
    gamma = widgets.FloatSlider(
        value=0.1,
        min=0.01,
        max=0.5,
        step=0.01,
        description='γ (выздоровление):',
        style={'description_width': 'initial'}
    )
    
    mu = widgets.FloatSlider(
        value=0.01,
        min=0.001,
        max=0.1,
        step=0.001,
        description='μ (смертность):',
        style={'description_width': 'initial'},
        disabled=True
    )
    
    days = widgets.IntSlider(
        value=160,
        min=30,
        max=365,
        step=10,
        description='Дней моделирования:',
        style={'description_width': 'initial'}
    )
    
    # Функция обновления доступности виджетов в зависимости от выбранной модели
    def update_widgets(*args):
        if model_selector.value == 'SIR':
            initial_exposed.disabled = True
            sigma.disabled = True
            mu.disabled = True
        elif model_selector.value == 'SEIR':
            initial_exposed.disabled = False
            sigma.disabled = False
            mu.disabled = True
        elif model_selector.value == 'SIRD':
            initial_exposed.disabled = True
            sigma.disabled = True
            mu.disabled = False
    
    model_selector.observe(update_widgets, 'value')
    
    # Кнопка запуска
    run_button = widgets.Button(
        description='Запустить моделирование',
        button_style='success',
        tooltip='Нажмите для запуска моделирования'
    )
    
    # Выходные данные
    output = widgets.Output()
    
    # Функция для моделирования
    def run_simulation(b):
        with output:
            clear_output(wait=True)
            
            N = population.value
            I0 = initial_infected.value
            E0 = initial_exposed.value if model_selector.value == 'SEIR' else 0
            R0_initial = 0
            D0 = 0
            S0 = N - I0 - R0_initial - E0 - D0
            
            t = np.linspace(0, days.value, days.value)
            
            # Выбор модели и решение системы дифференциальных уравнений
            if model_selector.value == 'SIR':
                y0 = [S0, I0, R0_initial]
                solution = odeint(sir_model, y0, t, args=(beta.value, gamma.value, N))
                S, I, R = solution.T
                E = None
                D = None
                
            elif model_selector.value == 'SEIR':
                y0 = [S0, E0, I0, R0_initial]
                solution = odeint(seir_model, y0, t, args=(beta.value, sigma.value, gamma.value, N))
                S, E, I, R = solution.T
                D = None
                
            elif model_selector.value == 'SIRD':
                y0 = [S0, I0, R0_initial, D0]
                solution = odeint(sird_model, y0, t, args=(beta.value, gamma.value, mu.value, N))
                S, I, R, D = solution.T
            
            # Вычисление R0
            if model_selector.value == 'SIRD':
                r0 = calculate_r0(beta.value, gamma.value, mu.value, 'SIRD')
            else:
                r0 = calculate_r0(beta.value, gamma.value, model=model_selector.value)
            
            # Создание графика
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
            
            # График динамики популяции
            ax1.plot(t, S, 'b-', label='Восприимчивые (S)', linewidth=2)
            if E is not None:
                ax1.plot(t, E, 'y-', label='Экспонированные (E)', linewidth=2)
            ax1.plot(t, I, 'r-', label='Инфицированные (I)', linewidth=2)
            ax1.plot(t, R, 'g-', label='Выздоровевшие (R)', linewidth=2)
            if D is not None:
                ax1.plot(t, D, 'k-', label='Умершие (D)', linewidth=2)
            
            ax1.set_xlabel('Дни')
            ax1.set_ylabel('Количество людей')
            ax1.set_title(f'{model_selector.value} модель эпидемии\nR₀ = {r0:.2f}')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            
            # График фазового портрета
            ax2.plot(I, S, 'purple', linewidth=2)
            ax2.set_xlabel('Инфицированные (I)')
            ax2.set_ylabel('Восприимчивые (S)')
            ax2.set_title('Фазовый портрет (S-I)')
            ax2.grid(True, alpha=0.3)
            
            # Добавление информации о пике эпидемии
            peak_infected = np.max(I)
            peak_day = np.argmax(I)
            ax2.scatter(I[peak_day], S[peak_day], color='red', s=100, zorder=5)
            ax2.annotate(f'Пик: день {peak_day}\nI = {peak_infected:.0f}', 
                        xy=(I[peak_day], S[peak_day]), 
                        xytext=(I[peak_day]+100, S[peak_day]-500),
                        arrowprops=dict(arrowstyle='->', color='red'))
            
            plt.tight_layout()
            plt.show()
            
            # Вывод статистики
            print(f"\n--- Статистика {model_selector.value} модели ---")
            print(f"Базовое репродуктивное число (R₀): {r0:.2f}")
            print(f"Максимальное число зараженных: {peak_infected:.0f} человек (день {peak_day})")
            print(f"Общее число выздоровевших: {R[-1]:.0f} человек")
            if D is not None:
                print(f"Общее число умерших: {D[-1]:.0f} человек")
                print(f"Летальность: {(D[-1]/(R[-1]+D[-1])*100):.2f}%")
            print(f"Осталось восприимчивых: {S[-1]:.0f} человек")
    
    run_button.on_click(run_simulation)
    
    # Создание панели управления
    controls = widgets.VBox([
        model_selector,
        population,
        initial_infected,
        initial_exposed,
        beta,
        sigma,
        gamma,
        mu,
        days,
        run_button
    ])
    
    # Создание основного интерфейса
    dashboard = widgets.HBox([controls, output])
    
    return dashboard

# Создание и отображение интерактивной панели
dashboard = create_simulation()
display(dashboard)

# Добавление пояснений
print("\n" + "="*50)
print("Инструкция по использованию:")
print("="*50)
print("1. Выберите модель эпидемии (SIR, SEIR или SIRD)")
print("2. Настройте параметры с помощью ползунков:")
print("   • Население - общая численность популяции")
print("   • Начальные условия - количество зараженных/экспонированных")
print("   • β - коэффициент контактности (скорость распространения)")
print("   • σ - скорость перехода из экспонированного состояния в зараженное (для SEIR)")
print("   • γ - скорость выздоровления")
print("   • μ - коэффициент смертности (для SIRD)")
print("3. Нажмите кнопку для запуска моделирования")
print("4. Наблюдайте динамику эпидемии на графиках")
print("\nR₀ - базовое репродуктивное число:")
print("• R₀ < 1 - эпидемия затухает")
print("• R₀ = 1 - эпидемия стабильна")
print("• R₀ > 1 - эпидемия растет")


Инструкция по использованию:
1. Выберите модель эпидемии (SIR, SEIR или SIRD)
2. Настройте параметры с помощью ползунков:
   • Население - общая численность популяции
   • Начальные условия - количество зараженных/экспонированных
   • β - коэффициент контактности (скорость распространения)
   • σ - скорость перехода из экспонированного состояния в зараженное (для SEIR)
   • γ - скорость выздоровления
   • μ - коэффициент смертности (для SIRD)
3. Нажмите кнопку для запуска моделирования
4. Наблюдайте динамику эпидемии на графиках

R₀ - базовое репродуктивное число:
• R₀ < 1 - эпидемия затухает
• R₀ = 1 - эпидемия стабильна
• R₀ > 1 - эпидемия растет
